# 03a — Scenario Selection Baseline (Classical)

This notebook is the first step of Phase 3. It turns the Phase 2 scenario
artifacts into a concrete **trial selection problem** and solves it with a
simple classical baseline.

We work with the scenario tables produced in Phase 2:

- `data/scenarios/scenario_A_trials.csv`
- `data/scenarios/scenario_B_trials.csv`

For a chosen scenario (e.g., Scenario B), we define a trial selection problem:

> Select a subset of trials to **maximize total benefit_score**  
> subject to **total estimated_trial_cost ≤ budget**.

This notebook:

1. Loads a scenario table and inspects key fields.
2. Defines a configurable budget.
3. Implements a simple greedy baseline:
   - Sort trials by benefit_score / estimated_trial_cost.
   - Pick trials in that order until the budget is exhausted.
4. Produces summary metrics and a selected subset of trials that can later be
   used for QUBO design and quantum experiments.


In [1]:
# ============================================================
# Cell 1 — Load Scenario A/B tables
# ============================================================

from pathlib import Path
import pandas as pd

def log(msg: str) -> None:
    print(msg)

SCENARIO_A_PATH = Path("data/scenarios/scenario_A_trials.csv")
SCENARIO_B_PATH = Path("data/scenarios/scenario_B_trials.csv")

# Load both if available, so we can easily switch between them
scenario_tables = {}

if SCENARIO_A_PATH.exists():
    scenario_A = pd.read_csv(SCENARIO_A_PATH)
    scenario_tables["A"] = scenario_A
    log(f"[Cell 1] Loaded Scenario A with shape {scenario_A.shape}")
else:
    log(f"[Cell 1] WARNING: {SCENARIO_A_PATH} not found.")

if SCENARIO_B_PATH.exists():
    scenario_B = pd.read_csv(SCENARIO_B_PATH)
    scenario_tables["B"] = scenario_B
    log(f"[Cell 1] Loaded Scenario B with shape {scenario_B.shape}")
else:
    log(f"[Cell 1] WARNING: {SCENARIO_B_PATH} not found.")

if not scenario_tables:
    raise FileNotFoundError(
        "[Cell 1] No scenario tables found. "
        "Run 02d_define_scenarios_A_B.ipynb first."
    )

# For now, default to Scenario B if present, otherwise Scenario A
if "B" in scenario_tables:
    scenario_id = "B"
else:
    scenario_id = "A"

trials_scenario = scenario_tables[scenario_id].copy()
log(f"[Cell 1] Using Scenario {scenario_id} as the working scenario.")
trials_scenario.head()


[Cell 1] Loaded Scenario A with shape (16672, 13)
[Cell 1] Loaded Scenario B with shape (10268, 13)
[Cell 1] Using Scenario B as the working scenario.


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score
0,NCT00003042,Chemotherapy and Stem Cell Transplantation in ...,"Active, not recruiting",Phase 2,['Breast Cancer'],['filgrastim' 'cisplatin' 'cyclophosphamide' '...,['United States'],City of Hope Medical Center,City of Hope Medical Center,Global / Multi-Region,2.0,1.0,0.6
1,NCT00007150,Treatment of Hemochromatosis,"Active, not recruiting",Phase 2,['Hemochromatosis'],['Phlebotomy'],['United States'],National Institutes of Health Clinical Center ...,National Institutes of Health Clinical Center ...,Global / Multi-Region,2.0,1.0,0.6
2,NCT00018057,Study of Neuro-Cognitive Correlates of Pediatr...,Recruiting,Phase 2,['Anxiety Disorders' 'Major Depressive Disorder'],['Attention Bias Modification Training' 'Fluox...,['United States'],National Institute of Mental Health (NIMH),National Institute of Mental Health (NIMH),Global / Multi-Region,2.0,1.0,0.6
3,NCT00044304,Tyrosine Kinase Inhibition to Treat Myeloid Hy...,Recruiting,Phase 2,['Eosinophilic Myeloid Neoplasm' 'Hypereosinop...,['Imatinib' 'Ruxolitinib'],['United States'],National Institute of Allergy and Infectious D...,National Institute of Allergy and Infectious D...,Global / Multi-Region,2.0,1.0,0.6
4,NCT00070499,Imatinib Mesylate or Dasatinib in Treating Pat...,"Active, not recruiting",Phase 2,"['Chronic Myeloid Leukemia, BCR-ABL1 Positive']",['Dasatinib' 'Imatinib Mesylate' 'Laboratory B...,['Canada' 'United States'],National Cancer Institute (NCI),National Cancer Institute (NCI),Global / Multi-Region,2.0,1.0,0.6


### What Cell 1 Just Did

This step loaded the scenario tables produced in Phase 2 and chose one working
scenario for the rest of the notebook.

Specifically, it:

- Read `data/scenarios/scenario_A_trials.csv` and
  `data/scenarios/scenario_B_trials.csv` (when present).
- Stored them in a small dictionary so we can easily switch between them.
- Selected a default working scenario:
  - Prefer Scenario **B** if it exists (high-benefit, cost-conscious),
  - Otherwise fall back to Scenario **A**.
- Copied the chosen scenario into `trials_scenario` and previewed the first
  few rows.

From here on, `trials_scenario` is the table we will optimize over when we
define and solve the budgeted trial selection problem.


In [2]:
# ============================================================
# Cell 2 — Inspect scenario fields and key distributions
# ============================================================

log(f"[Cell 2] Basic info for Scenario {scenario_id} trials:")

log(f"Shape: {trials_scenario.shape}")
log(f"Columns: {list(trials_scenario.columns)}")

required_cols = [
    "nct_id",
    "benefit_score",
    "estimated_trial_cost",
    "phase",
    "overall_status",
]

missing = [c for c in required_cols if c not in trials_scenario.columns]
if missing:
    log(f"[Cell 2] WARNING: Missing required columns: {missing}")

display(
    trials_scenario[
        ["nct_id", "phase", "overall_status", "estimated_trial_cost", "benefit_score"]
    ].head(10)
)

log("[Cell 2] Summary of estimated_trial_cost:")
display(trials_scenario["estimated_trial_cost"].describe())

log("[Cell 2] Summary of benefit_score:")
display(trials_scenario["benefit_score"].describe())


[Cell 2] Basic info for Scenario B trials:
Shape: (10268, 13)
Columns: ['nct_id', 'brief_title', 'overall_status', 'phase', 'conditions', 'interventions', 'location_countries', 'lead_sponsor', 'lead_sponsor_norm', 'region_label', 'estimated_trial_cost', 'enrollment_feasibility_score', 'benefit_score']


,nct_id,phase,overall_status,estimated_trial_cost,benefit_score
0,NCT00003042,Phase 2,"Active, not recruiting",2.0,0.6
1,NCT00007150,Phase 2,"Active, not recruiting",2.0,0.6
2,NCT00018057,Phase 2,Recruiting,2.0,0.6
3,NCT00044304,Phase 2,Recruiting,2.0,0.6
4,NCT00070499,Phase 2,"Active, not recruiting",2.0,0.6
5,NCT00076310,Phase 2,"Active, not recruiting",2.0,0.6
6,NCT00077285,Phase 2,"Active, not recruiting",2.0,0.6
7,NCT00080756,Phase 2,"Active, not recruiting",2.0,0.6
8,NCT00082706,Phase 2,"Active, not recruiting",2.0,0.6
9,NCT00085982,Phase 2,"Active, not recruiting",2.0,0.6


[Cell 2] Summary of estimated_trial_cost:


count    10268.0
mean         2.0
std          0.0
min          2.0
25%          2.0
50%          2.0
75%          2.0
max          2.0
Name: estimated_trial_cost, dtype: float64

[Cell 2] Summary of benefit_score:


count    10268.0
mean         0.6
std          0.0
min          0.6
25%          0.6
50%          0.6
75%          0.6
max          0.6
Name: benefit_score, dtype: float64

### What Cell 2 Just Did

This step performed a quick structural and distributional check on the working
scenario table to ensure it is ready for optimization.

Concretely, it:

- Logged the overall shape of `trials_scenario` and listed all columns.
- Verified that key fields needed for selection are present:
  - `nct_id`
  - `benefit_score`
  - `estimated_trial_cost`
  - `phase`
  - `overall_status`
- Displayed a small preview of these core columns for a handful of trials.
- Summarized the distributions of:
  - `estimated_trial_cost` (cost scale and spread)
  - `benefit_score` (range and central tendency)

If these checks look reasonable, it gives us confidence that the scenario
table contains the right ingredients for a meaningful budgeted selection
problem in the next step.


In [ ]:
# ============================================================
# Cell 3 — Greedy selection under a budget constraint
# ============================================================
#
# Objective:
#   - Maximize total benefit_score
# Constraint:
#   - Total estimated_trial_cost <= BUDGET
#
# Method:
#   - Compute benefit_per_cost = benefit_score / estimated_trial_cost
#   - Sort descending by benefit_per_cost
#   - Pick trials greedily until adding the next would exceed the budget.

import numpy as np

if missing:
    log("[Cell 3] ERROR: Cannot run selection; required columns missing.")
else:
    # You can tweak this as needed; we start with a fraction of total cost
    total_cost = trials_scenario["estimated_trial_cost"].sum()
    default_budget = 0.1 * total_cost  # 10% of total scenario cost

    BUDGET = default_budget
    log(f"[Cell 3] Using budget BUDGET = {BUDGET:,.2f} "
        f"(~10% of total scenario cost).")

    # Avoid division by zero by replacing 0 cost with a tiny epsilon
    cost = trials_scenario["estimated_trial_cost"].replace(0, np.finfo(float).eps)
    benefit = trials_scenario["benefit_score"].fillna(0.0)

    trials_scenario = trials_scenario.copy()
    trials_scenario["benefit_per_cost"] = benefit / cost

    # Sort by benefit_per_cost descending
    trials_sorted = trials_scenario.sort_values(
        "benefit_per_cost", ascending=False
    ).reset_index(drop=True)

    selected_rows = []
    running_cost = 0.0
    running_benefit = 0.0

    for idx, row in trials_sorted.iterrows():
        c = float(row["estimated_trial_cost"])
        b = float(row["benefit_score"])

        if running_cost + c <= BUDGET:
            selected_rows.append(row)
            running_cost += c
            running_benefit += b

    if not selected_rows:
        log("[Cell 3] WARNING: Greedy selection picked no trials under the budget.")
        selected_df = trials_sorted.head(0).copy()
    else:
        selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

    log(f"[Cell 3] Greedy selection picked {len(selected_df)} trials.")
    log(f"[Cell 3] Total selected cost:    {running_cost:,.2f}")
    log(f"[Cell 3] Total selected benefit: {running_benefit:,.4f}")

    display(
        selected_df[
            ["nct_id", "phase", "overall_status",
             "estimated_trial_cost", "benefit_score", "benefit_per_cost"]
        ].head(20)
    )


### What Cell 3 Just Did

This step turned the scenario table into a concrete optimization problem and
solved it with a simple greedy baseline.

The problem was defined as:

> Maximize total `benefit_score`  
> subject to total `estimated_trial_cost` ≤ **BUDGET**

To do this, the cell:

1. Computed a default budget as ~10% of the total cost of all trials in the
   working scenario.
2. Calculated a `benefit_per_cost` ratio for each trial:
   \[
   \text{benefit\_per\_cost} = \frac{\text{benefit\_score}}{\text{estimated\_trial\_cost}}
   \]
   (with a tiny epsilon to avoid dividing by zero).
3. Sorted trials from highest to lowest `benefit_per_cost`.
4. Walked down this sorted list and **greedily selected** trials, adding each
   one as long as the cumulative cost stayed within the budget.
5. Reported:
   - How many trials were selected,
   - The total cost of the selected set, and
   - The total benefit_score achieved.
6. Displayed a sample of the selected trials including:
   `nct_id`, `phase`, `overall_status`, `estimated_trial_cost`,
   `benefit_score`, and `benefit_per_cost`.

This gives us a clear, interpretable **classical baseline** for the trial
selection problem that future QUBO and quantum approaches can be compared
against.


In [ ]:
# ============================================================
# Cell 4 — Persist greedy baseline selection
# ============================================================

SOLUTIONS_DIR = Path("data/scenarios")
SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)

solution_path = SOLUTIONS_DIR / f"scenario_{scenario_id}_greedy_selection.csv"

if "selected_df" in globals():
    selected_df.to_csv(solution_path, index=False)
    log(
        f"[Cell 4] Wrote greedy baseline selection for Scenario {scenario_id} "
        f"to {solution_path} with shape {selected_df.shape}"
    )
else:
    log("[Cell 4] No selected_df found; nothing was written.")


### What Cell 4 Just Did

This step saved the greedy baseline solution so it can be reused by later
notebooks without rerunning the selection logic.

Specifically, it:

- Ensured that `data/scenarios/` exists.
- Wrote the selected trials to:
  - `data/scenarios/scenario_B_greedy_selection.csv`  
    (or the Scenario A equivalent, depending on which scenario was active).
- Preserved the **full set of columns** for each selected trial, not just
  `nct_id`, so downstream notebooks can:
  - Inspect sponsor, region, cost, feasibility, and benefit fields, and
  - Directly connect this classical solution to QUBO and quantum experiments.

This CSV now serves as the **baseline solution artifact** for Phase 3: any
future QUBO/QAOA notebooks can load it, compare against it, and measure how
well quantum or hybrid approaches perform relative to this greedy benchmark.


## Notebook Summary — Classical Scenario Selection Baseline

In this notebook we converted the Phase 2 scenario tables into a concrete
selection problem and solved it with a simple classical baseline:

1. Loaded Scenario A/B trial tables from `data/scenarios/`.
2. Focused on one scenario (default: Scenario B) as the working set.
3. Defined a budgeted selection objective:
   - Maximize total `benefit_score`
   - Subject to `total estimated_trial_cost ≤ BUDGET`
4. Implemented a greedy baseline:
   - Ranked trials by `benefit_score / estimated_trial_cost`
   - Selected trials in that order until the budget was exhausted.
5. Saved the resulting selection to:
   - `data/scenarios/scenario_B_greedy_selection.csv` (or Scenario A equivalent)

This gives us:
- A clear, testable **classical baseline** for trial selection, and
- A concrete subset of trials that we can now use when designing the QUBO
  and building quantum experiments in subsequent Phase 3 notebooks.
